# Intuition test: probability

Questions come first. For each one, write your prediction in the cell marked `Prediction`, then run the code cell below it. Answers are at the end, after every question. Do not scroll down early.

Seeded and self-contained.

In [ ]:
import numpy as np
from scipy import stats

rng = np.random.default_rng(42)
np.set_printoptions(precision=4, suppress=True)

## Q1. The base rate

A disease affects 0.5% of people. The test catches 99% of sick people and wrongly flags 5% of healthy people. You test positive. Roughly what is the chance you are sick: above 90%, about 50%, or under 15%?

Prediction: *(write here)*

In [ ]:
prior, sens, fpr = 0.005, 0.99, 0.05
posterior = sens * prior / (sens * prior + fpr * (1 - prior))
print("P(sick | positive):", posterior)
people = rng.random(1_000_000) < prior
pos = rng.random(1_000_000) < np.where(people, sens, fpr)
print("simulated         :", people[pos].mean())

## Q2. Updating a belief

You start with a flat Beta(1, 1) prior on a coin's bias and see 7 heads in 10 flips. What is the posterior mean? Now a friend starts with Beta(20, 20) and sees the same flips. Whose posterior mean is higher, and by roughly how much? How many flips would the friend need before the two are within 0.01 of each other?

Prediction: *(write here)*

In [ ]:
def post_mean(a: float, b: float, k: int, n: int) -> float:
    return (a + k) / (a + b + n)


print("flat      :", post_mean(1, 1, 7, 10))
print("friend    :", post_mean(20, 20, 7, 10))
for n in (10, 100, 300, 1000):
    k = round(0.7 * n)
    gap = post_mean(1, 1, k, n) - post_mean(20, 20, k, n)
    print(f"n={n:>5}: gap={gap:.4f}")

## Q3. Which loss, and why

For each case, name the loss you would use and the noise assumption behind it.

a) Predict house prices; errors look symmetric and bell-shaped.
b) Predict whether an email is spam.
c) Predict delivery times; most errors are small but a few are huge.
d) Classify a photo into one of 10 animals.
e) Predict a count of customer calls per hour.

Prediction: *(write here)*

In [ ]:
# Check c: with a few huge errors, which estimate of the typical delivery time is stable?
times = np.concatenate([rng.normal(30, 3, 95), rng.normal(300, 50, 5)])
print("squared error picks (mean)  :", times.mean())
print("absolute error picks (median):", np.median(times))

# Check e: the Poisson NLL is minimised at the sample mean
calls = rng.poisson(6, 500)
grid = np.linspace(3, 9, 601)
nll = [-stats.poisson.logpmf(calls, lam).sum() for lam in grid]
print("Poisson MLE:", grid[int(np.argmin(nll))], " sample mean:", calls.mean())

## Q4. Averages

You average 25 noisy measurements, each with standard deviation 10. What is the standard deviation of the average? How many measurements would you need to make it 1?

Prediction: *(write here)*

In [ ]:
means = rng.normal(0, 10, size=(20_000, 25)).mean(axis=1)
print("sd of the mean of 25:", means.std())
means = rng.normal(0, 10, size=(20_000, 100)).mean(axis=1)
print("sd of the mean of 100:", means.std())

## Answers

**Q1.** About 9%, under 15%. Out of 1000 people, 5 are sick and about 4.95 test positive. Of the 995 healthy, about 50 test positive. So roughly 5 of 55 positives are sick. Rare disease plus a modest false positive rate means most positives are false.

**Q2.** Flat: 8/12 = 0.667. Friend: 27/50 = 0.54. The friend's prior is worth 40 pseudo-flips, against 10 real ones. The gap is still 0.02 at 300 flips and falls under 0.01 only by about 1000, when the real data is 25 times the prior.

**Q3.**
a) Squared error, from Gaussian noise. The MLE is the mean.
b) Binary cross-entropy (log loss), from a Bernoulli outcome.
c) Absolute error or Huber loss. Heavy tails mean the Gaussian assumption is wrong, and the median ignores the huge errors where the mean does not.
d) Softmax with categorical cross-entropy.
e) Poisson negative log-likelihood, since the target is a count.

**Q4.** $10/\sqrt{25}=2$. For a standard deviation of 1 you need $n=100$. Four times the data halves the error, which is why precision gets expensive.